# Use-case: WWTD-2025 (What Would Trump Do?)

Generate a forecasting dataset about Trump's actions, decisions, and statements using the LightningRod SDK. This example showcases dataset generation, preparation with SDK utils, and training results from our experiments—including evaluation with and without context.

In [6]:
%pip install lightningrod-ai python-dotenv pandas

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [7]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for Trump-related forecasting.

In [8]:
instructions = """
Generate binary forecasting questions about Trump's actions, decisions, positions, and statements.
Questions should be diverse, related to the content, and should evenly cover the full range from very likely to very unlikely.
Horizon: outcomes should be known within 2 months of the question date, and may be known much sooner.
Criteria: binary outcome, exact dates, self-contained, verifiable via web search, newsworthy.
"""

good_examples = [
    "Will Trump impose 25% tariffs on all goods from Canada by February 1, 2025?",
    "Will Trump issue pardons to January 6 defendants within his first week in office?",
    "Will Pete Hegseth be confirmed as Secretary of Defense by February 15, 2025?",
    "Will Trump sign an executive order to keep TikTok operational in the US by January 31, 2025?",
    "Will Kash Patel be confirmed as FBI Director by March 1, 2025?",
]

bad_examples = [
    "Will Trump do something controversial? (too vague)",
    "Will Trump be in the news? (obvious)",
    "Will tariffs be imposed? (needs specifics)",
]

In [9]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2025, 1, 1),
        end_date=datetime(2026, 1, 1),
        interval_duration_days=7,
        search_query=[
            "Donald Trump domestic policy agenda",
            "Donald Trump trade and tariff actions",
            "Donald Trump foreign policy decisions",
            "Donald Trump interviews and press appearances",
            "Donald Trump lawsuits and court rulings",
        ],
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=20,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=1,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. Use `max_questions` to limit the run for testing.

In [10]:
dataset = lr.transforms.run(pipeline, max_questions=500, name="WWTD-2025")

samples = dataset.download()
pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

Output()

178 samples (46.1% valid)


## Prepare the dataset

Use SDK utils to filter valid samples, deduplicate, and split into train/test sets. We filter by `date_close <= today` to only include questions that have already resolved.

In [ ]:
from lightningrod.training import prepare_for_training

train, test = prepare_for_training(
    samples,
    answer_type,
    test_size=0.2,
    split_strategy="temporal",
    include_assistant=True,
    days_to_resolution_range=(1, 60),  # horizon within 2 months
)

for name, data in [("Train", train), ("Test", test)]:
    yes_count = sum(s["label"] for s in data)
    print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")

Train: 39 rows, 12.8% yes
Test: 13 rows, 38.5% yes


In [12]:
def _display_head(data, name, n=5):
    if not data:
        print(f"{name}: no rows")
        return
    df = pd.DataFrame(data[:n])
    cols = [
        "question_text", "prediction_date", "date_close", "resolution_date",
        "answer_type", "answer_parser_type", "reward_function_type",
        "label", "label_confidence", "prompt"
    ]
    display_cols = [c for c in cols if c in df.columns]
    print(f"{name} (head):")
    display(df[display_cols])

_display_head(train, "Train")
_display_head(test, "Test")

Train (head):


,question_text,prediction_date,date_close,resolution_date,answer_type,label,label_confidence,prompt
0,Will Donald Trump hold a formal press conferen...,2025-06-04T00:00:00,2025-07-31T00:00:00,2025-07-31T00:00:00,binary,0.0,0.9,"[{'role': 'user', 'content': 'QUESTION: Will D..."
1,Will more than three Democratic-led US states ...,2025-06-04T00:00:00,2025-07-04T00:00:00,2025-07-04T00:00:00,binary,0.0,0.9,"[{'role': 'user', 'content': 'QUESTION: Will m..."
2,Will Donald Trump issue a proclamation expandi...,2025-06-04T00:00:00,2025-08-01T00:00:00,2025-08-01T00:00:00,binary,0.0,0.9,"[{'role': 'user', 'content': 'QUESTION: Will D..."
3,Will Donald Trump add Egypt to the list of cou...,2025-06-04T00:00:00,2025-08-01T00:00:00,2025-06-09T00:00:00,binary,0.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will D..."
4,Will Donald Trump sign an order to remove Hait...,2025-06-04T00:00:00,2025-07-15T00:00:00,2025-06-04T00:00:00,binary,0.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will D..."


Test (head):


,question_text,prediction_date,date_close,resolution_date,answer_type,label,label_confidence,prompt
0,Will Nicki Minaj perform the national anthem a...,2026-01-19T00:00:00,2026-01-20T00:00:00,2026-01-20T00:00:00,binary,0.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will N..."
1,Will Donald Trump grant executive clemency to ...,2026-01-19T00:00:00,2026-03-01T00:00:00,2026-02-17T00:00:00,binary,1.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will D..."
2,Will Donald Trump attend the opening ceremony ...,2026-01-19T00:00:00,2026-02-07T00:00:00,2026-02-06T00:00:00,binary,0.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will D..."
3,Will JD Vance officially announce a 2028 presi...,2026-01-19T00:00:00,2026-03-01T00:00:00,2026-03-01T00:00:00,binary,0.0,0.9,"[{'role': 'user', 'content': 'QUESTION: Will J..."
4,Will an 'international stabilization force' in...,2026-01-19T00:00:00,2026-02-28T00:00:00,2026-02-19T00:00:00,binary,1.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will a..."


## Uploading the dataset to HuggingFace

Once we have a training-ready dataset, we can push it to Hugging Face for sharing or downstream use.

In [13]:
%pip install datasets -q

from datasets import Dataset, DatasetDict
from lightningrod.utils import config

dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test": Dataset.from_list(test),
})
print(f"Train: {len(dataset['train'])} rows, Test: {len(dataset['test'])} rows")
print("Columns:", dataset["train"].column_names[:8], "...")

DATASET_PATH = f"{config.get_config_value('HF_USERNAME')}/wwtd-forecasting-demo"
dataset.push_to_hub(DATASET_PATH, token=config.get_config_value("HF_ACCESS_TOKEN"))


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Train: 58 rows, Test: 17 rows
Columns: ['question_text', 'date_close', 'event_date', 'resolution_criteria', 'prediction_date', 'label', 'answer_type', 'label_confidence'] ...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/bart/wwtd-forecasting-demo/commit/cd7bfd6d7addc58cf2c3ac8f6677219a1ce91a91', commit_message='Upload dataset', commit_description='', oid='cd7bfd6d7addc58cf2c3ac8f6677219a1ce91a91', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bart/wwtd-forecasting-demo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bart/wwtd-forecasting-demo'), pr_revision=None, pr_num=None)

## Model Training

We used the generated dataset above to fine-tune a forecasting model via RL on 2,790 questions, surpassing GPT-5 performance.

**For more details on methods, results, and data:**
- **[Trump-Forecaster Model](https://huggingface.co/LightningRodLabs/Trump-Forecaster)**
- **[Trump-Forecaster Dataset](https://huggingface.co/datasets/LightningRodLabs/WWTD-2025)**

![Brier Skill Score](https://huggingface.co/datasets/LightningRodLabs/WWTD-2025/resolve/main/brier_skill_score.png)

**Coming Soon:** Seamlessly generate datasets, fine-tune, and evaluate your own forecasting models end-to-end on the Lightningrod platform.
 
👉 [Sign up to get early access and updates.](https://lightningrod.ai/)